In [5]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

# 1. On importe TES classes
from PyBH.SurvivalAnalysis.SurvivalAnalysis import SurvivalAnalysis
from PyBH.SurvivalAnalysis.pymc_models import Weibull # ou Cox, au choix !

# 2. Le générateur de données
def generer_dataset_etudiants(n_etudiants=1000):
    np.random.seed(42)
    sommeil = np.random.normal(6.5, 1.2, n_etudiants).clip(3, 10)
    cafes = np.random.poisson(3, n_etudiants).clip(0, 10)
    soirees = np.random.poisson(1.5, n_etudiants).clip(0, 5)
    sport = np.random.poisson(1, n_etudiants).clip(0, 7)
    bde = np.random.binomial(1, 0.3, n_etudiants)

    risque_cache = (-0.6*(sommeil-6.5) + 0.15*cafes + 0.4*soirees - 0.4*sport + 0.3*bde)
    taux_risque = np.exp(risque_cache) * 0.03
    
    u = np.random.uniform(0, 1, n_etudiants)
    temps_survie = -np.log(u) / taux_risque
    
    events = (temps_survie <= 24).astype(int)
    temps_finaux = np.minimum(temps_survie, 24)
    
    return pd.DataFrame({
        'heures_sommeil': np.round(sommeil, 1),
        'nb_cafes': cafes,
        'soirees_semaine': soirees,
        'seances_sport': sport,
        'membre_asso': bde,
        'time': np.maximum(0.1, np.round(temps_finaux, 1)), 
        'event': events
    })

# 3. Création et entraînement
df_etudiants = generer_dataset_etudiants(1000)
print("Données étudiantes générées !")

modele_survie = Weibull()
print("Entraînement du modèle...")
sa = SurvivalAnalysis(
    model=modele_survie, 
    data=df_etudiants, 
    time_col='time', 
    event_col='event',
    progressbar=False 
)
print("Modèle prêt.")

Initializing NUTS using jitter+adapt_diag...


Données étudiantes générées !
Entraînement du modèle...
   -> Mode: Bayesian (PyMC)


Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha, beta]
Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.


Modèle prêt.


In [6]:
# --- INTERFACE ---
clear_output(wait=True)
print("🎮 BIENVENUE DANS LE SIMULATEUR DE SURVIE ÉTUDIANTE 🎮")
print("Découvre combien de semaines tu vas tenir avant le burn-out...\n")
time.sleep(1)

try:
    # Les questions posées au visiteur
    sommeil = float(input("🛌 Combien d'heures dors-tu par nuit en moyenne ? (ex: 6) : "))
    cafes = int(input("☕ Combien de cafés bois-tu par jour ? (ex: 3) : "))
    soirees = int(input("🍻 Combien de soirées fais-tu par semaine ? (ex: 2) : "))
    sport = int(input("🏃‍♂️ Combien de séances de sport par semaine ? (ex: 1) : "))
    bde = int(input("🎉 Es-tu membre d'une asso très active (BDE/BDS...) ? (1 = Oui, 0 = Non) : "))
    
    print("\n🔮 L'IA calcule ton espérance de vie...")
    time.sleep(1.5) # Petit suspense
    
    # Nettoyage de l'écran pour n'afficher que le résultat
    clear_output(wait=True)
    print("📊 RÉSULTAT DE TA SURVIE :")
    
    # Formatage des réponses pour le modèle
    profil_visiteur = np.array([[sommeil, cafes, soirees, sport, bde]])
    
    # Tracé des courbes
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # 1. Courbe du Visiteur (en rouge pétant)
    sa.plot_survival_function(X_pred=profil_visiteur, ax=ax, label="TA SURVIE 🔴", color="red", linewidth=4)
    
    # 2. Courbe de l'Étudiant Moyen (en gris discret)
    sa.plot_survival_function(ax=ax, label="Étudiant Moyen ⚪", color="gray", linestyle="--", alpha=0.7)
    
    # Décoration du graphique
    ax.set_title("Probabilité de rester en forme ce semestre", fontsize=16, fontweight='bold')
    ax.set_xlabel("Semaines de cours (0 à 24)", fontsize=12)
    ax.set_ylabel("Probabilité de survie (%)", fontsize=12)
    ax.set_xlim(0, 24)
    ax.set_ylim(0, 1.05)
    
    # On ajoute une ligne verticale pour marquer la date des partiels
    ax.axvline(x=12, color='orange', linestyle=':', label='Semaine des Partiels')
    ax.legend(fontsize=12)
    
    plt.show()

except ValueError:
    print("\n⚠️ Oups ! Tu dois rentrer des chiffres (ex: 3). Relance la cellule !")

🎮 BIENVENUE DANS LE SIMULATEUR DE SURVIE ÉTUDIANTE 🎮
Découvre combien de semaines tu vas tenir avant le burn-out...


⚠️ Oups ! Tu dois rentrer des chiffres (ex: 3). Relance la cellule !
